# Tutorial: Fundamentos de SimPy para a Aula D10

**Público:** alunos com Python básico que estão começando em simulação.

**Pré-requisitos:** variáveis, funções, `for`, `if`, noção básica de filas.

**Objetivos de aprendizagem:**

- entender o que é simulação de eventos discretos;
- entender o papel de `Environment`, `Process`, `Event` e `timeout`;
- saber quando usar `Resource`, `PriorityResource`, `Store`, `FilterStore` e `Container`;
- chegar nos exemplos da D10 já sabendo o que cada peça do SimPy faz.


## Roteiro

1. O que o SimPy modela.
2. Como o relógio da simulação avança.
3. O que é um processo em SimPy.
4. Como pensar em recursos e filas.
5. Como escolher a estrutura correta para cada problema hospitalar.
6. Exercícios rápidos de fixação.


## 1. O que é simulação de eventos discretos?

Em SimPy, o tempo **não anda segundo a segundo** como em um relógio real. Ele avança de **evento em evento**.

Exemplos de eventos em saúde:

- chegada de paciente;
- início da triagem;
- fim da consulta;
- liberação de leito;
- retorno de ambulância.

A palavra **discreto** significa que o modelo presta atenção nesses marcos importantes, e não em todos os instantes intermediários.


In [1]:
import simpy

print(f"Versão do SimPy: {simpy.__version__}")

Versão do SimPy: 4.1.1


## 2. O relógio do SimPy: `Environment`

O objeto `Environment` é o coração da simulação.

Ele faz três coisas:

- guarda o tempo atual em `env.now`;
- agenda eventos futuros;
- executa a fila de eventos com `env.run()`.

A ideia mental correta é esta: **o ambiente é o relógio e a agenda do sistema**.


In [2]:
import simpy

env = simpy.Environment()
print(f"Tempo inicial: {env.now}")

Tempo inicial: 0


## 3. Processo = uma história ao longo do tempo

Em SimPy, um processo normalmente é uma função com `yield`.

Cada `yield` diz ao ambiente: **pare aqui e só me retome quando este evento acontecer**.

O comando mais comum é:

`yield env.timeout(t)`

que significa: **espere `t` unidades de tempo simuladas**.


In [3]:
import simpy


def paciente_exemplo(env):
    print(f"{env.now:02.0f} min | paciente chegou")
    yield env.timeout(5)
    print(f"{env.now:02.0f} min | paciente terminou a etapa")


env = simpy.Environment()
env.process(paciente_exemplo(env))
env.run()

00 min | paciente chegou
05 min | paciente terminou a etapa


### Leitura do resultado

Observe o raciocínio:

- o paciente chega no tempo `0`;
- o processo pede uma espera de `5` minutos simulados;
- o relógio salta para `5`;
- a próxima linha é executada.

Esse é o mecanismo base de quase tudo em SimPy.


## 4. Recursos: quando existe fila

Quando várias entidades disputam algo limitado, usamos recursos.

A pergunta correta é sempre:

**"O que está sendo disputado no sistema?"**

Exemplos:

- triagem disputa enfermeiro;
- consulta disputa médico;
- internação disputa leito;
- remoção disputa ambulância;
- tarefa operacional disputa equipe disponível.


## 5. Mapa mental das estruturas principais

| Estrutura          | Quando usar                          | Exemplo hospitalar                  |
| ------------------ | ------------------------------------ | ----------------------------------- |
| `Resource`         | capacidade simultânea sem prioridade | balcão, sala, médico                |
| `PriorityResource` | fila com prioridade                  | Manchester, urgência                |
| `Store`            | objetos nomeados                     | leito UTI-A, UTI-B                  |
| `FilterStore`      | objetos nomeados com filtro          | ambulância básica vs UTI            |
| `Container`        | quantidade agregada, sem identidade  | número de profissionais disponíveis |

Essa tabela é um dos pontos mais importantes da aula.


## 6. Como decidir entre `Store` e `Container`

Use esta regra prática:

- se a pergunta é **"quantos há disponíveis?"**, use `Container`;
- se a pergunta é **"qual item específico está disponível?"**, use `Store` ou `FilterStore`.

Exemplo:

- "Tenho 4 enfermeiros disponíveis agora" -> `Container`.
- "Qual ambulância tipo UTI está livre?" -> `FilterStore`.


## 7. Erros conceituais mais comuns

1. Usar `Resource` quando o item deveria ter identidade própria.
2. Esquecer que prioridade menor significa atendimento antes em `PriorityResource`.
3. Achar que `env.run(until=t)` executa tudo que está exatamente em `t`.
4. Misturar regra de negócio com impressão de tela sem separar o raciocínio.


## 8. Exercícios de fixação

1. Se eu preciso modelar leitos nomeados, uso `Resource` ou `Store`?
2. Se quero representar apenas a quantidade total de técnicos em plantão, uso `Container` ou `FilterStore`?
3. O que significa `yield env.timeout(10)` em termos do relógio da simulação?


In [4]:
# Respostas esperadas (deixe escondido na aula se quiser):
# 1. Store, porque cada leito é um objeto identificável.
# 2. Container, porque o que importa é a quantidade agregada.
# 3. O processo pausa e só continua 10 unidades de tempo simuladas depois.

## 9. Ponte para os próximos notebooks

Nos próximos exemplos, você vai ver essas ideias aplicadas em cenários hospitalares didáticos:

- triagem com prioridade clínica;
- fila por leito com timeout;
- fluxo completo do cadastro à alta;
- escala por capacidade agregada;
- despacho por tipo de ambulância.

A partir daqui, o foco deixa de ser "o que é a ferramenta" e passa a ser "como modelar corretamente o sistema".
